# Depth-Conditioned Gaussian V_θ on Fock-PARFLM v2.1 — TinyStories

## Goal

Evaluate the **depth-conditioned multi-context Gaussian V_θ** (the same
V_θ architecture used at d=384/d=768 on OpenWebText) as a drop-in
replacement for the MLP V_θ in the **Multi-channel ξ Fock-PARFLM v2.1**
(`FockMultiXiPARFLM`) on TinyStories at d=256.

This enables a direct comparison with the **SQ3 quadratic** structured
V_θ (`colab_fock_multixi_structured_vtheta.ipynb`) and the **MLP**
baseline, all at d=256 on the same dataset:

| V_θ variant | Bounded? | Wells | Key property |
|---|---|---|---|
| MLP (baseline) | No | — | Arbitrary learned landscape |
| SQ3 Quadratic | No (clamped) | K mixture | Locally quadratic, `curv_max` safety |
| **Gaussian (this)** | **Yes** | K per head × N heads | Structurally bounded force |

## Architecture

- d=256, L=8, M=16 registers, fixed_gamma=0.3, logfreq mass
- V_θ: `DepthConditionedMultiContextGaussianVTheta` (shared bank + per-layer depth codes)
- 4 xi channels, 4 heads × 8 wells/head = 32 total attractors
- PARFLM: structural_competitive routing, top_k=8, Gumbel gates
- Fock v2.1: stack discipline, reverse channel, per-register τ/keys
- Two-stage causal probes (architectural + trained-scale)
- Periodic checkpointing + resume from latest checkpoint

## Companion documents

- `companion_notes/Fock-PARFLM_Causal_Leak_Audit_Results.md` — causal leak analysis
- `companion_notes/Geodesic_Preservation_Experiment.md` — theory and design


In [ ]:
# ── Cell 0: Configuration ──────────────────────────────────────────
import math as _math

SEED = 0

# ── Architecture ──
D              = 256
L              = 8
VOCAB_SIZE     = 50257
MAX_LEN        = 1024
DT             = 1.0
FIXED_GAMMA    = 0.30

# ── V_theta: Depth-Conditioned Multi-Context Gaussian ──
V_THETA_VARIANT             = 'gaussian'
V_THETA_N_HEADS             = 4        # one bank per xi channel
V_THETA_WELLS_PER_HEAD      = 8        # Gaussian wells per bank
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02

# ── Xi channels ──
XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
XI_LEARNABLE   = True

# ── PARFLM ──
V_PHI_KIND     = 'structural_competitive'
TOP_K          = 8

# ── Training ──
STEPS          = 16_000
BATCH          = 16
GRAD_ACCUM     = 1
BLOCK          = 512
LR             = 5e-4
WD             = 0.01
WARMUP         = 400
GRAD_CLIP      = 1.0
LAMBDA_V       = 1e-2

# ── Evaluation ──
EVAL_INTERVAL  = 400
EVAL_ITERS     = 40
LOG_INTERVAL   = 50

# ── Checkpointing ──
CHECKPOINT_INTERVAL = 1000

# ── Causal probes ──
CAUSAL_PROBE_INTERVAL       = 4000
TRAINED_LEAK_PROBE_INTERVAL = 8000
TRAINED_LEAK_PROBE_K        = 256
TRAINED_LEAK_PROBE_PAIRS    = 2

print(f'Gaussian V_theta on TinyStories d={D} L={L}')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} total attractors')
print(f'  Depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  xi_channels={XI_CHANNELS}  alpha_inits={XI_ALPHA_INITS}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}  lr={LR}')
print(f'  lambda_V={LAMBDA_V}  grad_clip={GRAD_CLIP}')


In [ ]:
# ── Cell 1: Environment + Drive Mount ──────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')

    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_gaussian_vtheta_tinystories')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)

    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    RESULTS_DIR = GDRIVE_ROOT / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'fock_gaussian_vtheta_tinystories'
    for d in [DATA_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

RUN_DIR = RESULTS_DIR / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_DIR    = {DATA_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'RUN_DIR     = {RUN_DIR}')


In [ ]:
# ── Cell 2: GPU Check + Imports ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

print('Model imports OK')


In [ ]:
# ── Cell 3: Data Loading (TinyStories) ─────────────────────────────
from data_module import get_batch, load_tiny_stories

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

rng = np.random.default_rng(SEED)


In [ ]:
# ── Cell 4: Model Builder + Gaussian V_theta ──────────────────────

# ── Logfreq surprisal ──
LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_tinystories.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_FILE}')

print(f'Logfreq: {LOGFREQ_FILE}')

# ── Build model ──
torch.manual_seed(SEED)

cfg = FockMultiXiPARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=1024, v_depth=3, dt=DT,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_FILE),
    logfreq_init_alpha=0.1,
    init_gamma=1.0,
    fixed_gamma=FIXED_GAMMA,
    causal_force=True,
    ln_after_step=True,
    xi_channels=XI_CHANNELS,
    xi_alpha_inits=XI_ALPHA_INITS,
    xi_learnable=XI_LEARNABLE,
    xi_alpha_init_mode='explicit',
    v_phi_kind=V_PHI_KIND,
    v_phi_phi_hidden=128,
    v_phi_theta_hidden=128,
    top_k=TOP_K,
    score_head_hidden=32,
    gumbel_tau_init=1.0,
    gumbel_tau_min=0.3,
    gumbel_noise=True,
    use_gathered_v_phi=True,
    use_layer_checkpoint=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    fock_version='v2',
    n_registers=16,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    creation_gate_hidden=64,
    stack_discipline=True,
    d_k=64,
    tau_create_init=8.0,
    reverse_channel=True,
    per_register_tau=True,
    per_register_keys=True,
    ortho_register_init=True,
    prefix_causal_registers=True,
)
model = FockMultiXiPARFLM(cfg).to(DEVICE)

n_total_before = sum(p.numel() for p in model.parameters())
n_v_theta_before = sum(p.numel() for p in model.V_theta.parameters())
print(f'Before swap: total={n_total_before:,}  V_theta(MLP)={n_v_theta_before:,}')

# ── Swap in Gaussian V_theta ──
from model_gaussian_vtheta import (
    DepthConditionedMultiContextGaussianVTheta,
    install_depth_routing,
)

import math as _m
model.V_theta = DepthConditionedMultiContextGaussianVTheta(
    d=D,
    K=V_THETA_WELLS_PER_HEAD,
    n_ctx=V_THETA_N_HEADS,
    n_layers=L,
    w_scale=1.0,
    init_log_precision=-_m.log(D),
    precision_max=2.0 / D,
    code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
).to(DEVICE)
install_depth_routing(model)

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'After swap:')
print(f'  total params   = {n_total:,}')
print(f'  V_theta params = {n_v_theta:,}  ({n_v_theta/n_total*100:.1f}%)')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells '
      f'= {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors, depth-conditioned')
print(f'  xi_alpha init: {model.xi_alpha_values()}')
print(f'Model builder OK')


In [ ]:
# ── Cell 5: Training + Evaluation Helpers ──────────────────────────

EVAL_MICRO_BATCH = 4


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(model, x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.float().reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp
    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate_model():
    model.eval()
    micro = min(EVAL_MICRO_BATCH, BATCH)
    n_micro = max(1, BATCH // micro)
    losses = []
    for _ in range(EVAL_ITERS):
        micro_losses = []
        for _m in range(n_micro):
            xb, yb = get_batch(val_ids, micro, BLOCK, rng)
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            with torch.enable_grad():
                _, loss = model(x, y)
            micro_losses.append(loss.item())
            del loss, x, y
        losses.append(float(np.mean(micro_losses)))
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    model.train()
    return float(np.mean(losses))


print('Training helpers OK')


In [ ]:
# ── Cell 6: Two-Stage Causal Probes ────────────────────────────────

def run_causal_probe(step_num):
    """Stage 1: lightweight architectural causal probe (CPU, float64).

    Builds a tiny model with the same structural features, opens the
    reverse channel fully, and checks that perturbing future tokens
    produces exactly zero change in logits at earlier positions.
    Returns (passed: bool, max_delta: float).
    """
    import math as _math
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=1, dt=0.1,
        mass_mode='global', causal_force=True,
        ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=False, xi_alpha_init_mode='explicit',
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)

    _probe_model.V_theta = DepthConditionedMultiContextGaussianVTheta(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0 / _PROBE_D, code_init_std=0.02,
    )
    install_depth_routing(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        for n, p in _probe_model.named_parameters():
            if 'reverse_channel_scale' in n:
                p.fill_(5.0)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    _max_delta = 0.0
    for mode_name, use_train in [('eval', False), ('train', True)]:
        if use_train:
            _probe_model.train()
            torch.manual_seed(99)
        else:
            _probe_model.eval()
        with torch.enable_grad():
            _la = _probe_model(_x1)[0].detach()
        if use_train:
            torch.manual_seed(99)
        with torch.enable_grad():
            _lb = _probe_model(_x2)[0].detach()
        delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())
        _max_delta = max(_max_delta, delta)

    _passed = (_max_delta == 0.0)

    del _probe_model, _la, _lb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    """Stage 2: trained-scale leak probe + honest PPL on the live model."""
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} — running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)

    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK, batch=BATCH, device=DEVICE)

    model.train()

    result = {
        'step': step_num,
        'event': 'trained_leak_probe',
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


print('Two-stage causal probes OK')


In [ ]:
# ── Cell 7: Training Loop ──────────────────────────────────────────

resume_step = 0
_latest_ckpt_path = RUN_DIR / 'ckpt_latest.pt'
_best_ckpt_path = RUN_DIR / 'ckpt_best.pt'

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)

if _latest_ckpt_path.exists():
    ckpt = torch.load(_latest_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt:
        try:
            opt.load_state_dict(ckpt['optimizer_state_dict'])
            print(f'Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'[info] Optimizer state incompatible: {e}')
    resume_step = ckpt.get('step', 0)
    print(f'Resumed from step {resume_step:,}  '
          f'(PPL {ckpt.get("val_ppl", "?")})  [{_latest_ckpt_path.name}]')
    del ckpt

best_val_ppl = float('inf')
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored best PPL from previous session: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] Could not read best checkpoint: {e}')

# Advance RNG past completed steps for deterministic data ordering
if resume_step > 0:
    for _ in range(resume_step * GRAD_ACCUM):
        get_batch(train_ids, BATCH, BLOCK, rng)

log_path = RUN_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

SPIKE_THRESHOLD  = 500.0
SPIKE_COOLDOWN   = 20
_last_spike_step = -10**9

val_ppl = best_val_ppl


def _log_write(record):
    log_f.write(json.dumps(record) + '\n')
    log_f.flush()


model.train()
t0 = time.time()
t_session = time.time()

for step in range(resume_step, STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    opt.zero_grad(set_to_none=True)
    step_loss_ntp = 0.0
    step_v_reg = 0.0
    step_loss_total = 0.0

    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        step_loss_ntp   += loss_ntp.item() / GRAD_ACCUM
        step_v_reg      += v_reg.item()    / GRAD_ACCUM
        step_loss_total += loss.item()     / GRAD_ACCUM

    grad_norm = nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    ).item()
    opt.step()

    # Clamp Gaussian precision after each step
    for bank in model.V_theta.banks:
        if hasattr(bank, 'clamp_params'):
            bank.clamp_params()

    # Spike detection
    if (grad_norm > SPIKE_THRESHOLD
            and (step - _last_spike_step) >= SPIKE_COOLDOWN):
        _last_spike_step = step
        print(f'\n[spike] step {step+1}: pre-clip grad={grad_norm:.1f}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}')
        _log_write({
            'step': step + 1, 'event': 'grad_spike',
            'pre_clip_grad_norm': round(grad_norm, 2),
            'ntp': round(step_loss_ntp, 4),
            'v_reg': round(step_v_reg, 4),
        })

    # Causal probes
    if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
        _cp_passed, _cp_delta = run_causal_probe(step + 1)
        _log_write({
            'step': step + 1, 'event': 'causal_probe',
            'causal_probe_passed': _cp_passed,
            'causal_probe_max_delta': _cp_delta,
        })

    if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
        _tlp = run_trained_leak_probe(step + 1)
        _log_write(_tlp)

    # Logging
    if (step + 1) % LOG_INTERVAL == 0 or step == resume_step:
        elapsed = time.time() - t_session
        steps_done = step + 1 - resume_step
        sec_per_step = elapsed / max(steps_done, 1)
        remaining = (STEPS - step - 1) * sec_per_step
        alphas_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())
        print(f'step {step+1:>5}/{STEPS}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}  '
              f'lr={lr_at(step):.2e}  grad={grad_norm:.3f}  '
              f'gamma={model.gamma.item():.3f}  '
              f'alpha=[{alphas_str}]  '
              f'{elapsed:.0f}s (~{remaining/60:.1f}m remaining)')
        _log_write({
            'step': step + 1, 'train_loss': step_loss_ntp,
            'v_reg': step_v_reg, 'total_loss': step_loss_total,
            'lr': lr_at(step), 'grad_norm': grad_norm,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
        })

    # Evaluation
    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate_model()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        best_marker = '  *** NEW BEST ***' if is_best else ''
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best={best_val_ppl:.2f}{best_marker}')
        _log_write({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        })
        if is_best:
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'gamma': model.gamma.item(),
                'xi_alphas': model.xi_alpha_values(),
                'v_theta_variant': V_THETA_VARIANT,
            }, str(_best_ckpt_path))

    # Periodic checkpoint
    if (step + 1) % CHECKPOINT_INTERVAL == 0 or (step + 1) == STEPS:
        _ckpt_ppl = val_ppl if (step + 1) % EVAL_INTERVAL == 0 else best_val_ppl
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'step': step + 1,
            'val_ppl': _ckpt_ppl,
            'best_val_ppl': best_val_ppl,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
            'v_theta_variant': V_THETA_VARIANT,
        }, str(_latest_ckpt_path))
        print(f'  [ckpt] saved {_latest_ckpt_path.name} at step {step+1}')

log_f.close()
print(f'\nTraining done.  total wall = {time.time()-t_session:.0f}s  '
      f'final_ppl = {val_ppl:.2f}  best_ppl = {best_val_ppl:.2f}')


In [ ]:
# ── Cell 8: Training Curve ─────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
            except Exception:
                pass

if eval_entries:
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Gaussian V_theta ({V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD})',
            linewidth=1.5)
    ax.axhline(y=8.95, color='red', linestyle='--', alpha=0.7,
               label='Fock-PARFLM v2.1 MLP baseline (8.95 PPL)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Gaussian V_theta on Fock-PARFLM v2.1 — TinyStories d={D}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUN_DIR / 'training_curve_gaussian.png', dpi=150)
    plt.show()
    print(f'Saved: {RUN_DIR / "training_curve_gaussian.png"}')
else:
    print('No eval data to plot.')


In [ ]:
# ── Cell 9: V_theta Landscape Diagnostics ─────────────────────────
v_samples = []
model.eval()
for _ in range(10):
    xb, _ = get_batch(val_ids, min(BATCH, 4), BLOCK, rng)
    x = torch.from_numpy(xb).to(DEVICE)
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_samples.append(V_vals.detach().cpu().numpy().ravel())

v_all = np.concatenate(v_samples)
ls = {
    'mean': float(v_all.mean()),
    'std': float(v_all.std()),
    'min': float(v_all.min()),
    'max': float(v_all.max()),
    'range': float(v_all.max() - v_all.min()),
}
print(f'V_theta landscape stats:')
for k, v in ls.items():
    print(f'  {k:6s}: {v:.4f}')

ls_path = RUN_DIR / 'landscape_stats_gaussian.json'
with open(ls_path, 'w') as f:
    json.dump(ls, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(v_all, bins=80, edgecolor='none', alpha=0.8)
ax.axvline(ls['mean'], color='red', linestyle='--', alpha=0.6, label=f'mean={ls["mean"]:.2f}')
ax.set_xlabel('V_theta(xi, h)')
ax.set_ylabel('count')
ax.set_title(f'Gaussian V_theta distribution (d={D}, {V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD} wells)')
ax.legend()
plt.tight_layout()
fig.savefig(RUN_DIR / 'v_theta_hist_gaussian.png', dpi=150)
plt.show()
model.train()
print(f'Saved: {RUN_DIR / "v_theta_hist_gaussian.png"}')


In [ ]:
# ── Cell 10: Summary ───────────────────────────────────────────────
summary_path = RUN_DIR / 'summary_gaussian.md'
with open(summary_path, 'w') as f:
    f.write(f'# Depth-Conditioned Gaussian V_theta on Fock-PARFLM v2.1 — TinyStories\n\n')
    f.write(f'| Setting | Value |\n')
    f.write(f'|---------|-------|\n')
    f.write(f'| V_theta variant | Gaussian (depth-conditioned multi-context) |\n')
    f.write(f'| V_theta heads | {V_THETA_N_HEADS} |\n')
    f.write(f'| Wells per head | {V_THETA_WELLS_PER_HEAD} |\n')
    f.write(f'| Total attractors | {V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} |\n')
    f.write(f'| Depth conditioning | {V_THETA_DEPTH_CONDITION} |\n')
    f.write(f'| xi_channels | {XI_CHANNELS} |\n')
    f.write(f'| lambda_V | {LAMBDA_V} |\n')
    f.write(f'| d | {D} |\n')
    f.write(f'| L | {L} |\n')
    f.write(f'| steps | {STEPS} |\n')
    f.write(f'| best PPL | {best_val_ppl:.2f} |\n')
    f.write(f'| V_theta params | {n_v_theta:,} |\n')
    f.write(f'| total params | {n_total:,} |\n')
    f.write(f'| xi_alpha_init | {XI_ALPHA_INITS} |\n')
    f.write(f'\n## Reference\n\n')
    f.write(f'Fock-PARFLM v2.1 MLP baseline (K_xi=4, 16k steps): **8.95 PPL**\n')

print(f'Summary saved to {summary_path}')
print(f'\nFinal result: Gaussian V_theta = {best_val_ppl:.2f} PPL  '
      f'(Fock-PARFLM v2.1 MLP baseline = 8.95)')
